<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Filtering in Spatial Domain — Theory</b></h1>
</div>

## Theoretical Foundations

This notebook documents the mathematical model, estimation methods, numerical considerations, diagnostics, and limitations used in the laboratory.
### Technical Context

Spatial filters use a neighborhood around each pixel to suppress noise, enhance detail, or estimate local derivatives.

### Core Spatial Filtering Model

A linear spatial filter combines neighborhood samples with a kernel. In convolution, the kernel is flipped relative to correlation; nonlinear filters such as the median use an order statistic rather than a weighted sum.

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $I$ | input image |
| $K$ | filtering kernel |
| $(x,y)$ | image coordinate |
| $I*K$ | convolution |
| $\sigma$ | Gaussian standard deviation |
| $\nabla I$ | image gradient |

### Analytical Scope

Explain kernels, convolution/correlation, borders, smoothing, noise-specific denoising, sharpening, gradients, RGB filtering, numerical safety, and filter-selection trade-offs.


## 1. Data and Output Paths

The notebook locates the laboratory directory automatically by searching upward for both `data/` and `notebooks/`.

This makes execution robust whether VS Code starts the notebook from the repository root, the lab root, or the notebook directory.


## 2. Spatial Filtering Formulation

A **point operation** transforms one pixel using only its own value:

$$
g(x,y)=T(f(x,y))
$$

A **spatial neighborhood operation** uses nearby pixels:

$$
g(x,y)=T\left(\text{neighborhood around }(x,y)\right)
$$

This is the fundamental difference between the previous image-transformation laboratory and the present filtering laboratory.

Spatial filters can:

- suppress noise;
- blur small structures;
- preserve or destroy edges;
- emphasize rapid intensity changes;
- sharpen an image;
- estimate local derivatives.

### Deeper understanding

For a linear shift-invariant filter with kernel $h$, 2-D convolution is

$$
g(x,y)=\sum_m\sum_n h(m,n)\,f(x-m,y-n).
$$

The output at each location is therefore a weighted combination of a local neighborhood. Linearity means

$$
T(af_1+bf_2)=aT(f_1)+bT(f_2),
$$

and shift invariance means that translating the input translates the output by the same amount. These properties explain why convolution has a direct frequency-domain interpretation.


## 3. Kernel Properties and Normalization

A **kernel** (also called a mask or filter) is a small matrix of weights.

For a linear $3\times3$ filter:

$$
K=
\begin{bmatrix}
k_{-1,-1} & k_{0,-1} & k_{1,-1}\\
k_{-1,0}  & k_{0,0}  & k_{1,0}\\
k_{-1,1}  & k_{0,1}  & k_{1,1}
\end{bmatrix}
$$

Important kernel properties include:

- **size** — e.g. 3×3, 5×5, 9×9;
- **anchor / center** — location aligned with the current pixel;
- **weights** — determine how neighbors contribute;
- **sum of weights** — often controls response to constant regions;
- **symmetry** — important for smoothing and derivative behavior.

A normalized smoothing kernel usually has weights summing to 1.

### Deeper understanding

For smoothing kernels, unit DC gain is usually required:

$$
\sum_m\sum_n h(m,n)=1.
$$

This preserves constant regions because a constant input $f=c$ remains $g=c$. Derivative kernels instead typically sum to zero so that constant regions produce zero response.

Kernel symmetry controls phase behavior for many linear filters, and separability can reduce computational cost. If

$$
h(m,n)=a(m)b(n),
$$

a 2-D convolution can be computed as two 1-D convolutions, reducing the operation count substantially for large kernels.


## 4. Correlation vs Convolution

Correlation and convolution both slide a kernel across an image.

The difference is the kernel orientation.

### Correlation

For correlation, the kernel is used as written.

### Convolution

For convolution, the kernel is flipped horizontally and vertically before the sliding operation.

In 2-D:

$$
K_{\mathrm{conv}}(i,j)=K(-i,-j)
$$

If a kernel is symmetric, correlation and convolution produce the same result.

If it is asymmetric, they generally differ.

### Deeper understanding

Correlation is

$$
g_{\mathrm{corr}}(x,y)=\sum_m\sum_n h(m,n)f(x+m,y+n),
$$

whereas convolution flips the kernel before accumulation. For symmetric kernels $h(m,n)=h(-m,-n)$, the two operations are identical. For asymmetric derivative kernels, the sign or orientation of the response can change.

This distinction matters because some image-processing APIs implement correlation even when the operation is colloquially called filtering.


## 5. Direct Convolution Implementation

A direct convolution routine is implemented as a reference for validating library-based filtering.

The routine exposes the exact sequence of numerical operations:

```text
pad image
    ↓
extract neighborhood
    ↓
multiply by flipped kernel
    ↓
sum
    ↓
store output pixel
```


## 6. Border Handling

At the image border, part of the neighborhood lies outside the array.

A filtering algorithm must decide what values exist beyond the image.

Common strategies include:

- `constant` → fill with a fixed value, often 0;
- `nearest` → repeat the nearest border pixel;
- `reflect` → mirror the image around the edge;
- `mirror` → a related reflection convention;
- `wrap` → continue from the opposite side.

The border rule can change the numerical result.

### Deeper understanding

A finite image does not define samples outside its support, so filtering near boundaries requires an extension rule. Common choices include constant padding, replication, reflection, and periodic wrapping.

The boundary model changes the effective signal being filtered. Constant padding creates artificial discontinuities; replication assumes the edge value continues; reflection creates an even extension and often reduces edge discontinuity; periodic wrapping assumes opposite boundaries are adjacent. Border artifacts are therefore consequences of the assumed extension model, not of convolution alone.


## 7. Mean / Box Filtering

The $3\times3$ mean filter is:

$$
K=
\frac{1}{9}
\begin{bmatrix}
1&1&1\\
1&1&1\\
1&1&1
\end{bmatrix}
$$

Each output pixel is the arithmetic mean of its neighborhood.

The mean filter reduces local fluctuations, but it does not know whether a variation is noise or a real edge.

Therefore:

```text
larger averaging neighborhood
    → stronger smoothing
    → stronger edge/detail loss
```

### Deeper understanding

For a $k\times k$ box filter,

$$
h(m,n)=\frac{1}{k^2}.
$$

Averaging reduces zero-mean independent noise variance approximately in proportion to the number of averaged samples when local signal variation is small. However, it also attenuates high-frequency image detail. Increasing $k$ therefore trades noise suppression for spatial resolution.


## 8. Gaussian Filtering

A Gaussian filter gives larger weight to nearby pixels and smaller weight to distant pixels.

The continuous 2-D Gaussian is:

$$
G(x,y)=
\frac{1}{2\pi\sigma^2}
\exp\left(
-\frac{x^2+y^2}{2\sigma^2}
\right)
$$

The parameter $\sigma$ controls the spatial spread:

- small $\sigma$ → weak smoothing;
- large $\sigma$ → stronger smoothing.

Unlike a box filter, the weights vary smoothly with distance.

### Deeper understanding

The continuous 2-D Gaussian is

$$
G(x,y)=\frac{1}{2\pi\sigma^2}
\exp\left(-\frac{x^2+y^2}{2\sigma^2}\right).
$$

Its Fourier transform is also Gaussian, so it provides a smooth low-pass response without the sharp cutoff associated with strong ringing. The parameter $\sigma$ sets the smoothing scale, while kernel size should be large enough to capture most of the Gaussian mass, commonly several standard deviations around the center.


## 9. Noise-Model Dependence

The same filter should not be selected blindly for every degradation.

The provided Einstein images let us compare three important noise types:

- Gaussian;
- salt-and-pepper;
- speckle.

Their visual structure is different, so their preferred filters can also differ.


## 10. Median Filtering

The median filter is **nonlinear**.

For each neighborhood:

1. collect all pixel values;
2. sort them;
3. select the middle value;
4. assign that median to the output pixel.

Example neighborhood values:

```text
[20, 21, 20,
 22, 255, 19,
 20, 21, 20]
```

The value `255` is an impulse outlier.

The median remains near the normal neighborhood values instead of being pulled strongly upward.

### Deeper understanding

For a neighborhood $\mathcal{N}_{x,y}$,

$$
g(x,y)=\operatorname{median}\{f(i,j):(i,j)\in\mathcal{N}_{x,y}\}.
$$

Median filtering is nonlinear, so convolution theory does not apply. Its robustness comes from order statistics: isolated extreme values have little influence on the median unless they occupy a large fraction of the neighborhood. This makes it particularly effective for impulse noise while preserving step edges better than local averaging in many cases.


## 11. Bilateral Filtering

A bilateral filter smooths pixels using two notions of similarity:

1. **spatial similarity** — nearby pixels matter more;
2. **intensity similarity** — pixels with similar intensity matter more.

A simplified bilateral weight between a center pixel $p$ and a neighbor $q$ is:

$$
w(p,q)
=
\exp\left(
-\frac{\|p-q\|^2}{2\sigma_s^2}
\right)
\exp\left(
-\frac{|I_p-I_q|^2}{2\sigma_r^2}
\right)
$$

where:

- $\sigma_s$ controls spatial distance;
- $\sigma_r$ controls intensity/range similarity.

Because pixels across a strong edge have very different intensities, their contribution can be reduced.

### Deeper understanding

A bilateral filter combines spatial and radiometric weights:

$$
g(p)=
\frac{1}{W_p}
\sum_{q\in\mathcal{N}_p}
\exp\left(-\frac{\|p-q\|^2}{2\sigma_s^2}\right)
\exp\left(-\frac{|f(p)-f(q)|^2}{2\sigma_r^2}\right)
f(q),
$$

with normalization $W_p$ equal to the sum of the weights.

$\sigma_s$ controls spatial reach, while $\sigma_r$ controls how strongly intensity differences block averaging across edges. As $\sigma_r$ becomes very large, the bilateral filter approaches ordinary Gaussian spatial smoothing.


## 12. Quantitative Denoising Metrics

Visual inspection is essential, but quantitative metrics help compare outputs reproducibly.

For reference image $R$ and test image $T$:

### MAE

$$
\mathrm{MAE}
=
\frac{1}{N}\sum |R-T|
$$

### MSE

$$
\mathrm{MSE}
=
\frac{1}{N}\sum (R-T)^2
$$

### RMSE

$$
\mathrm{RMSE}=\sqrt{\mathrm{MSE}}
$$

### PSNR

For 8-bit images:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

Higher PSNR normally means lower pixel-wise error.


## 13. Edge Preservation as a Secondary Check

One simple way to inspect structural preservation is to compare gradient magnitude.

This is not a universal perceptual-quality metric, but it gives useful intuition about whether smoothing has weakened edges.


## 14. Sharpening with the Laplacian

Smoothing suppresses high local variation.

Sharpening does the opposite: it emphasizes rapid intensity changes.

The continuous Laplacian is:

$$
\nabla^2 f
=
\frac{\partial^2 f}{\partial x^2}
+
\frac{\partial^2 f}{\partial y^2}
$$

A common discrete 4-neighbor Laplacian kernel is:

$$
\begin{bmatrix}
0&1&0\\
1&-4&1\\
0&1&0
\end{bmatrix}
$$

Because Laplacian sign conventions differ between implementations, the sharpening formula must be checked carefully.

### Deeper understanding

The continuous Laplacian is

$$
\nabla^2 f=
\frac{\partial^2 f}{\partial x^2}
+
\frac{\partial^2 f}{\partial y^2}.
$$

Discrete Laplacian kernels approximate this second derivative and produce strong responses around rapid intensity changes. A common sharpening form is

$$
g=f-c\nabla^2 f,
$$

where the sign $c$ depends on the discrete kernel convention. Because second derivatives amplify high-frequency components strongly, Laplacian sharpening is highly sensitive to noise.


## 15. Unsharp Masking and High-Boost Filtering

Unsharp masking first creates a blurred version of the image.

The detail mask is:

$$
m=f-f_{\mathrm{blur}}
$$

Then the sharpened image is:

$$
g=f+k\,m
$$

where $k$ controls sharpening strength.

- $k=1$ → classical unsharp masking;
- $k>1$ → stronger high-boost sharpening.

### Deeper understanding

Let $\bar f$ be a low-pass version of $f$. The detail mask is

$$
m=f-\bar f.
$$

Unsharp masking uses

$$
g=f+k\,m.
$$

For $k=1$ this is standard unsharp enhancement; larger $k$ gives high-boost behavior. The method separates detail extraction from detail gain, making the sharpening strength explicit. The same detail mask contains both useful edges and high-frequency noise.


## 16. First Derivatives and Image Gradients

Edges correspond to rapid intensity variation.

The image gradient contains horizontal and vertical derivatives:

$$
\nabla f=
\begin{bmatrix}
G_x\\
G_y
\end{bmatrix}
$$

The gradient magnitude is:

$$
|\nabla f|
=
\sqrt{G_x^2+G_y^2}
$$

The gradient orientation is:

$$
\theta
=
\operatorname{atan2}(G_y,G_x)
$$

The Sobel operator combines differentiation with a small amount of local smoothing.

### Deeper understanding

The image gradient is

$$
\nabla f=
\begin{bmatrix}
G_x\\G_y
\end{bmatrix},
$$

with magnitude and orientation

$$
|\nabla f|=\sqrt{G_x^2+G_y^2},
\qquad
\theta=\operatorname{atan2}(G_y,G_x).
$$

The gradient points in the direction of greatest intensity increase and is normal to an ideal edge. Derivative operators amplify high frequencies, so smoothing and derivative estimation are often coupled in practical edge detectors.


## 17. Sobel vs Prewitt vs Scharr

Several derivative operators approximate image gradients.

### Prewitt

Uses simple derivative and smoothing weights.

### Sobel

Gives larger weight to the center row/column and is extremely common.

### Scharr

Uses coefficients designed to improve rotational symmetry for a 3×3 derivative operator.

No operator is universally best for every problem.

### Deeper understanding

Prewitt uses uniform smoothing perpendicular to the derivative direction. Sobel gives greater weight to the central row/column, providing slightly stronger smoothing. Scharr coefficients are designed to improve rotational symmetry for a $3\times3$ derivative operator.

The relevant comparison is therefore not only response magnitude but directional isotropy: an ideal gradient estimator should respond similarly to an edge regardless of its orientation.


## 18. Border Effects on a Real Image

Large kernels make border behavior easier to see.

We compare several padding modes using a 15×15 averaging filter.


## 19. Filtering RGB Images

A color image has shape:

```text
(H, W, 3)
```

A spatial filter should normally act over the two spatial dimensions while preserving the channel dimension.

For a Gaussian filter in SciPy, this can be expressed using:

```python
sigma=(sigma_y, sigma_x, 0)
```

The zero prevents smoothing across the channel axis.


## 20. Numerical Safety

Filtering often produces floating-point values or signed derivative responses.

Important rules:

1. Convert to floating point before operations that may become negative or exceed 255.
2. Do not immediately cast derivative images to `uint8`.
3. Clip only when producing a display/storage image that requires a bounded range.
4. Keep signed responses when the sign contains information.
5. Normalize kernels deliberately rather than automatically.

### Deeper understanding

Filtering should usually be performed in floating point when negative responses or values above the storage range are possible. Derivative and sharpening filters naturally produce signed values; performing them directly in unsigned 8-bit arithmetic can wrap or clip and destroy information.

The safe sequence is: convert to a computation dtype, perform the operation, inspect/normalize or clip according to the intended meaning, then convert back to a storage dtype only when necessary.


## 21. Filter Selection Criteria

A useful first decision table is:

| Situation | Reasonable first choice | Why |
|---|---|---|
| Mild additive Gaussian noise | Gaussian filter | smooth weighted averaging |
| Random impulse/salt-and-pepper noise | Median filter | rejects isolated extreme values |
| Noise with important edges | Bilateral filter | weights both distance and intensity similarity |
| General simple smoothing | Mean or Gaussian | simple neighborhood averaging |
| Blur requiring local contrast enhancement | Unsharp mask / Laplacian | emphasizes high local variation |
| Edge/gradient estimation | Sobel / Scharr / Prewitt | approximates spatial derivatives |

This table is a starting point, not a substitute for validation.


## 22. Standard Spatial-Filtering Workflow

A robust workflow is:

```text
1. Inspect image
    ↓
2. Identify degradation or objective
    ↓
3. Check dtype and intensity range
    ↓
4. Choose filter family
    ↓
5. Choose border handling
    ↓
6. Start with conservative parameters
    ↓
7. Apply filter in floating point when needed
    ↓
8. Compare visually
    ↓
9. Compare numerically if reference exists
    ↓
10. Check edge/detail preservation
    ↓
11. Tune parameters
    ↓
12. Validate and save result
```


## 23. Validation Checks

The following assertions verify key mathematical and implementation assumptions.

### Deeper understanding

Validation can exploit filter invariants: a normalized smoothing kernel should preserve a constant image; a zero-sum derivative kernel should return zero on a constant image; convolution output should remain finite and shape-compatible; and equivalent implementations should agree within numerical tolerance away from differing border conventions.

These checks verify the mathematical contract of the filter rather than only the absence of runtime errors.


## Technical Synthesis

Spatial filtering is governed by the local operator, the image boundary model, and the statistical structure of the degradation:

$$
\boxed{
\text{image}
\rightarrow
\text{neighborhood operator}
\rightarrow
\text{filtered response}
\rightarrow
\text{detail/noise trade-off}
\rightarrow
\text{quantitative + visual validation}
}
$$

Mean, Gaussian, median, bilateral, Laplacian, Sobel, Prewitt, and Scharr operators solve different local estimation problems. Kernel support, normalization, border handling, data type, and noise model must therefore be treated as experimental parameters rather than implementation details.

## Scope and Limitations

### Included

Convolution/correlation, smoothing, Gaussian/median/bilateral filtering, denoising metrics, Laplacian/unsharp/high-boost sharpening, gradients, borders, RGB filtering, and numerical safety.

### Not included

Explicit Fourier-domain filtering and segmentation pipelines.


## References

1. **R. C. Gonzalez and R. E. Woods**, *Digital Image Processing* — core reference for convolution, smoothing, sharpening, derivative operators, border behavior, and restoration metrics. [Companion site](https://www.imageprocessingplace.com/)
2. **OpenCV Documentation**, “Image Filtering” — aligned API/theory reference for linear filtering, bilateral filtering, kernels, border extrapolation, and multi-channel filtering. [OpenCV filtering reference](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html)
3. **SciPy Documentation**, `scipy.ndimage.gaussian_filter` — exact reference for Gaussian smoothing, sigma, derivative order, radius/truncation, and boundary modes. [SciPy API](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_filter.html)
4. **SciPy Documentation**, `scipy.ndimage.median_filter` — exact reference for neighborhood-based median filtering and footprint/window behavior. [SciPy API](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.median_filter.html)
5. **C. Tomasi and R. Manduchi**, “Bilateral Filtering for Gray and Color Images,” *ICCV*, 1998 — foundational reference for edge-preserving bilateral filtering. [DOI: 10.1109/ICCV.1998.710815](https://doi.org/10.1109/ICCV.1998.710815)
